### Links and Resources
| Concept / Command | Description |
|---|---|
| [Output modes](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#output-modes) | append, update, complete — what each emits |
| [Triggers](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#triggers) | processingTime, availableNow, once, continuous |
| [Checkpointing](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#recovering-from-failures-with-checkpointing) | Fault-tolerance and exactly-once recovery |
| [`Trigger.AvailableNow`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.streaming.DataStreamWriter.trigger.html) | Process all available data, then stop (batch-style streaming) |

In [2]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df, n=20):
    df.show(n, truncate=False)

# Output Modes, Triggers and Checkpoints

These three settings control *what* a streaming query emits, *when* it fires, and *how* it survives failures.

| Setting | What it controls |
|---|---|
| **Output mode** | Which rows are written to the sink each micro-batch |
| **Trigger** | How often a micro-batch is scheduled |
| **Checkpoint** | Where progress and state are stored for fault-tolerant restarts |

Understanding all three together is required to build streaming pipelines that are both correct and cost-efficient.

## 1. Output modes

| Mode | What is written each batch | Requires aggregation? | Requires watermark? |
|---|---|---|---|
| **`append`** | Only **new rows** added since the last batch | No | No (or yes for windowed aggs) |
| **`complete`** | The **entire** result table | Yes | No |
| **`update`** | Only rows whose value **changed** this batch | Yes | No |

**`append`** is the default and most common — every incoming event row is written once, never updated. It is the only mode supported for non-aggregated streaming DataFrames.

**`complete`** re-emits every aggregation key every batch. Only practical for small, bounded key spaces (a handful of users, not millions).

**`update`** emits only changed aggregation keys. Smaller writes than `complete`, but requires the sink to support upserts (Delta can, plain Parquet cannot).

In [2]:
import time, uuid
from pyspark.sql import functions as F

# rate-micro-batch (not plain rate) so availableNow can terminate the stream — see notebook 02.
events = (spark.readStream
    .format("rate-micro-batch").option("rowsPerBatch", 32).option("numPartitions", 1).load()
    .withColumnRenamed("timestamp", "event_time")
    .withColumnRenamed("value",     "event_id")
    .withColumn("user_id", (F.col("event_id") % 5).cast("int"))
    .withColumn("amount",  (F.rand() * 100 + 1).cast("int")))

CKPT_BASE = "/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints"

# append mode — non-aggregated stream: each row written once
q_append = (events
    .select("event_id", "user_id", "amount")
    .writeStream.format("memory").queryName("mode_append")
    .outputMode("append")
    .option("checkpointLocation", f"{CKPT_BASE}/mode_append_{uuid.uuid4().hex[:8]}")
    .trigger(availableNow=True)
    .start())

# complete mode — aggregated: whole table re-emitted every batch
q_complete = (events
    .withWatermark("event_time", "10 seconds")
    .groupBy("user_id")
    .agg(F.count("*").alias("events"), F.sum("amount").alias("total"))
    .writeStream.format("memory").queryName("mode_complete")
    .outputMode("complete")
    .option("checkpointLocation", f"{CKPT_BASE}/mode_complete_{uuid.uuid4().hex[:8]}")
    .trigger(availableNow=True)
    .start())

# update mode — aggregated: only changed keys per batch
q_update = (events
    .withWatermark("event_time", "10 seconds")
    .groupBy("user_id")
    .agg(F.count("*").alias("events"), F.sum("amount").alias("total"))
    .writeStream.format("memory").queryName("mode_update")
    .outputMode("update")
    .option("checkpointLocation", f"{CKPT_BASE}/mode_update_{uuid.uuid4().hex[:8]}")
    .trigger(availableNow=True)
    .start())

q_append.awaitTermination(); q_complete.awaitTermination(); q_update.awaitTermination()

print("append  — raw events (each row once, ever-growing table):")
spark.sql("SELECT COUNT(*) as total_rows FROM mode_append").show()

print("complete — full aggregation (all 5 user_ids, totals as of last batch):")
spark.sql("SELECT user_id, events, total FROM mode_complete ORDER BY user_id").show()

print("update — accumulation also shown (memory sink stores all updates):")
spark.sql("SELECT user_id, events, total FROM mode_update ORDER BY user_id").show()

append  — raw events (each row once, ever-growing table):
+----------+
|total_rows|
+----------+
|        32|
+----------+

complete — full aggregation (all 5 user_ids, totals as of last batch):
+-------+------+-----+
|user_id|events|total|
+-------+------+-----+
|      0|     7|  401|
|      1|     7|  320|
|      2|     6|  232|
|      3|     6|  302|
|      4|     6|  321|
+-------+------+-----+

update — accumulation also shown (memory sink stores all updates):
+-------+------+-----+
|user_id|events|total|
+-------+------+-----+
|      0|     7|  322|
|      1|     7|  407|
|      2|     6|  331|
|      3|     6|  322|
|      4|     6|  360|
+-------+------+-----+



## 2. Triggers — controlling micro-batch cadence

A trigger determines *when* a micro-batch fires. Every trigger is a trade-off between **latency** and **cost**.

| Trigger | How to set it | Behaviour |
|---|---|---|
| **`processingTime`** | `.trigger(processingTime="10 seconds")` | Fire every N seconds. If the previous batch took longer, fire immediately |
| **`availableNow`** | `.trigger(availableNow=True)` | Process all currently-available data in one or more batches, then **stop** |
| **`once`** *(deprecated)* | `.trigger(once=True)` | Same as `availableNow` but always one batch. Use `availableNow` instead |
| **`continuous`** | `.trigger(continuous="1 second")` | Experimental — ~1 ms latency, narrow operator support. Rarely needed |

**No trigger specified** → `processingTime="0 seconds"` — next batch starts immediately after the previous one finishes. Maximum throughput, maximum cost.

In [3]:
# This cell DEMONSTRATES the processingTime trigger but it is REJECTED on serverless
# (INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED). The trigger only works on a classic cluster.
# Wrapped in try/except so the rest of the notebook stays executable.
from pyspark.errors import AnalysisException

try:
    q = (events.select("event_id", "user_id")
         .writeStream.format("memory").queryName("trigger_pt")
         .outputMode("append")
         .option("checkpointLocation", f"{CKPT_BASE}/trigger_pt_{uuid.uuid4().hex[:8]}")
         .trigger(processingTime="3 seconds")
         .start())

    time.sleep(12)
    q.stop()

    # recentProgress shows one entry per batch — should see roughly 4 entries in 12 s
    print(f"Batches fired: {len(q.recentProgress)}")
    for p in q.recentProgress:
        print(f"  batchId={p['batchId']:>2}  "
              f"inputRows={p['numInputRows']:>4}  "
              f"trigger={p['trigger'].get('triggerExecution', {}).get('durationMs', '?')} ms")
except AnalysisException as e:
    print(f"processingTime trigger rejected on this cluster: {str(e).split(chr(10))[0]}")
    print("Use availableNow on serverless, or run this cell on a classic cluster.")
except Exception as e:
    print(f"rejected: {type(e).__name__}: {str(e)[:160]}")

processingTime trigger rejected on this cluster: [INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED] Trigger type ProcessingTime is not supported for this cluster type.
Use availableNow on serverless, or run this cell on a classic cluster.


## 3. `Trigger.AvailableNow` — batch-style streaming

`Trigger.AvailableNow` is the cleanest way to run a streaming query on a **schedule** rather than continuously.

**Use case:** process new files or Delta table changes every hour via a job scheduler (Databricks Workflows / LakeFlow Jobs), rather than keeping a stream running 24/7.

```
  09:00  job runs  → query reads all data since 08:00 → stops
  10:00  job runs  → query reads all data since 09:00 → stops
  11:00  ...
```

**Key behaviour:**
- Reads all data that exists at the moment the query starts.
- May fire multiple micro-batches (one per source file/partition), then terminates cleanly.
- The checkpoint records what was processed, so the next run picks up exactly where this one stopped — no duplicates, no gaps.

In [4]:
# Create a small Delta table as the source (file/Delta sources work with availableNow)
spark.sql(
    "CREATE OR REPLACE TABLE population_metrics.default.avail_now_src "
    "(event_id LONG, user_id INT, amount INT) USING DELTA"
)
spark.sql(
    "INSERT INTO population_metrics.default.avail_now_src VALUES "
    "(1,0,50),(2,1,30),(3,2,80),(4,3,15),(5,4,90),(6,0,45),(7,1,70)"
)

# availableNow: process all current data then stop automatically
q = (spark.readStream.format("delta")
     .table("population_metrics.default.avail_now_src")
     .writeStream.format("memory").queryName("avail_now_demo")
     .outputMode("append")
     .option("checkpointLocation", f"{CKPT_BASE}/avail_now_demo_{uuid.uuid4().hex[:8]}")
     .trigger(availableNow=True)
     .start())

q.awaitTermination(30)   # blocks until the query finishes naturally

print(f"Query active: {q.isActive}  (False = stopped cleanly)")
print(f"Batches processed: {len(q.recentProgress)}")
print("Rows processed:")
spark.sql("SELECT * FROM avail_now_demo ORDER BY event_id").show()

Query active: False  (False = stopped cleanly)
Batches processed: 1
Rows processed:
+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|       1|      0|    50|
|       2|      1|    30|
|       3|      2|    80|
|       4|      3|    15|
|       5|      4|    90|
|       6|      0|    45|
|       7|      1|    70|
+--------+-------+------+



## 4. Checkpoints — what they store

A checkpoint is a directory containing three sub-directories:

```
/my/checkpoint/location/
├── offsets/          ← what input has been READ (per batch)
│   ├── 0             ← batch 0: read files A, B (or Kafka offset 0–99)
│   ├── 1             ← batch 1: read files C (or Kafka offset 100–199)
│   └── ...
├── commits/          ← what has been WRITTEN and committed to the sink
│   ├── 0
│   ├── 1
│   └── ...
└── state/            ← in-memory state for stateful ops (windows, joins, dedup)
    └── 0/
        └── ...
```

**On restart, Spark reads the last offset and commit to determine where to resume:**
- If offset N was read but commit N is missing → batch N is replayed (source re-read, sink must be idempotent).
- If both offset N and commit N exist → batch N+1 starts from where offset N left off.

Delta Lake as a sink handles the re-play case automatically (idempotent writes via transaction IDs).

In [4]:
CATALOG   = "population_metrics.default"
CKPT_PATH = f"{CKPT_BASE}/ckpt_nb06_{uuid.uuid4().hex[:8]}"   # Unity Catalog volume — /tmp is rejected on serverless
SINK_TBL  = f"{CATALOG}.stream_ckpt_demo"

spark.sql(f"DROP TABLE IF EXISTS {SINK_TBL}")

# ── First run ─────────────────────────────────────────────────────────────────
q1 = (events.select("event_id", "user_id", "amount")
      .writeStream.format("delta")
      .option("checkpointLocation", CKPT_PATH)
      .outputMode("append")
      .trigger(availableNow=True)
      .toTable(SINK_TBL))

q1.awaitTermination()
count1     = spark.sql(f"SELECT COUNT(*) FROM {SINK_TBL}").collect()[0][0]
batch1     = q1.lastProgress["batchId"]

print(f"First run: {count1} rows written, last batchId = {batch1}")
print(f"Checkpoint stored at: {CKPT_PATH}")

NameError: name 'CKPT_BASE' is not defined

In [ ]:
# ── Restart with the SAME checkpoint ─────────────────────────────────────────
q2 = (events.select("event_id", "user_id", "amount")
      .writeStream.format("delta")
      .option("checkpointLocation", CKPT_PATH)   # same path — resumes from batch1+1
      .outputMode("append")
      .trigger(availableNow=True)
      .toTable(SINK_TBL))

q2.awaitTermination()
count2 = spark.sql(f"SELECT COUNT(*) FROM {SINK_TBL}").collect()[0][0]
batch2 = q2.lastProgress["batchId"]

print(f"After restart: {count2} rows total (added {count2 - count1} new rows)")
print(f"Resumed at batchId = {batch2}  (continues from {batch1 + 1}, no gap)")
print(f"No data lost. No data duplicated.")

## 5. Exploring the checkpoint directory

In [5]:
import subprocess, os

# List checkpoint directory to see offsets, commits, state
try:
    result = subprocess.run(
        ["find", CKPT_PATH, "-type", "f"],
        capture_output=True, text=True, timeout=10
    )
    files = [f for f in result.stdout.strip().split("\n") if f]
    for f in sorted(files)[:20]:
        print(f)
    if len(files) > 20:
        print(f"  ... ({len(files) - 20} more files)")
except Exception:
    # On Databricks serverless, /tmp is remote — use spark to list
    print("Checkpoint subdirectories:")
    for sub in ["offsets", "commits"]:
        try:
            items = spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem                 .get(spark.sparkContext._jsc.hadoopConfiguration())                 .listStatus(spark.sparkContext._jvm.org.apache.hadoop.fs.Path(f"{CKPT_PATH}/{sub}"))
            print(f"  {sub}/: {len(items)} files")
        except Exception as e:
            print(f"  {sub}/: {e}")

Checkpoint subdirectories:
  offsets/: [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `sparkContext` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.
  commits/: [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `sparkContext` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.


## 6. Production checkpoint rules

| Rule | Why |
|---|---|
| **One checkpoint per query** | Two queries sharing a checkpoint will corrupt each other's offsets |
| **Durable storage only** | Use Unity Catalog Volumes (`/Volumes/...`) or cloud storage (ADLS, S3). Never `/tmp` or local disk |
| **Never delete while the query runs** | Deleting mid-run causes data loss or duplication |
| **Deleting and restarting = starting over** | The query re-reads from the beginning of the source |
| **Schema changes may require a new checkpoint** | Adding columns to an aggregation, changing window size, or changing the source path breaks checkpoint compatibility |
| **Use unique, descriptive paths** | `/Volumes/catalog/schema/vol/checkpoints/pipeline_name/query_name` |

### When must you delete the checkpoint?
- The source schema changed in a breaking way.
- You intentionally want to re-process data from the beginning.
- You changed a stateful operation (window size, join key) that is incompatible with existing state.

## Takeaways

- **`append`** writes new rows once — correct for filters, projections, and (with watermark) windowed aggregations in final form.
- **`complete`** re-emits the whole result table every batch — only practical for small key spaces.
- **`update`** emits only changed aggregation rows — efficient, but requires an upsert-capable sink (Delta).
- **`processingTime`** fires on a fixed clock; no trigger fires on every completed batch (max throughput).
- **`Trigger.AvailableNow`** is the production pattern for scheduled "streaming on a cron" — processes all current data and stops cleanly, letting a job scheduler manage cadence.
- Every production query must have a **checkpoint location** on durable storage. One checkpoint per query; never share.
- Deleting a checkpoint = starting over. Schema changes may force a new checkpoint.